In [1]:
from mock_data import student, opportunities

In [43]:
import pandas as pd
import numpy as np

In [44]:
file_path = "Pakistan_Scholarships_Research-3.xlsx"

opportunities_raw = pd.read_excel(
    file_path,
    sheet_name="Scholarships"
)

In [45]:
opportunities_raw.head()

,Scholarship Name,Provider/Category,Amount / Coverage,Field of Study,Min CGPA / Eligibility,Province/Domicile,Level,Deadline (typical),Official Apply Link,Notes
0,HEC Ehsaas Undergraduate Scholarship,Government - HEC,100% tuition + monthly stipend,All disciplines,Need-based (NSER registered / low income),All provinces,Undergraduate,Announced yearly via HEC (check portal),https://ehsaas.hec.gov.pk,Largest need-based program in Pakistan; apply ...
1,HEC Need-Based Undergraduate Scholarship (UGSP),Government - HEC,100% tuition + stipend,All disciplines,Financial need assessed by university committee,All provinces (public HEC-recognized universit...,Undergraduate,Rolling / per semester,https://scholarship.hec.gov.pk,Only public sector universities; private/self-...
2,HEC Indigenous PhD Fellowship,Government - HEC,Monthly stipend + research funding,All disciplines,MPhil/MS with research aptitude,All provinces,PhD,Varies by cycle,https://hec.gov.pk/scholarships,Study within Pakistan at HEC-recognized univer...
3,HEC Overseas PhD Scholarships (various countries),Government - HEC,Full tuition + stipend abroad,All disciplines,"Strong academic record, MS/MPhil",All provinces,PhD,Varies,https://hec.gov.pk/scholarships,"Includes China, Hungary, France, Austria and o..."
4,Benazir Undergraduate Scholarship,Government (BISP-linked),Tuition support + stipend,All disciplines,Underprivileged/low-income background,All provinces,Undergraduate,Announced yearly,https://hec.gov.pk/scholarships,Jointly run with BISP for extremely low-income...


In [47]:
print(opportunities_raw.shape)

(38, 10)


In [48]:
print(opportunities_raw.columns.tolist())

['Scholarship Name', 'Provider/Category', 'Amount / Coverage', 'Field of Study', 'Min CGPA / Eligibility', 'Province/Domicile', 'Level', 'Deadline (typical)', 'Official Apply Link', 'Notes']


In [49]:
opportunities_raw.isnull().sum()

Scholarship Name          0
Provider/Category         0
Amount / Coverage         0
Field of Study            0
Min CGPA / Eligibility    0
Province/Domicile         0
Level                     0
Deadline (typical)        0
Official Apply Link       0
Notes                     0
dtype: int64

In [50]:
opportunities = opportunities_raw.rename(
    columns={
        "Scholarship Name": "name",
        "Provider/Category": "provider",
        "Amount / Coverage": "coverage",
        "Field of Study": "field_requirement_raw",
        "Min CGPA / Eligibility": "eligibility_raw",
        "Province/Domicile": "domicile_requirement_raw",
        "Level": "degree_level_raw",
        "Deadline (typical)": "deadline_raw",
        "Official Apply Link": "official_apply_link",
        "Notes": "notes"
    }
).copy()

In [51]:
opportunities.insert(
    0,
    "opportunity_id",
    range(1, len(opportunities) + 1)
)

In [52]:
opportunities.head()

,opportunity_id,name,provider,coverage,field_requirement_raw,eligibility_raw,domicile_requirement_raw,degree_level_raw,deadline_raw,official_apply_link,notes
0,1,HEC Ehsaas Undergraduate Scholarship,Government - HEC,100% tuition + monthly stipend,All disciplines,Need-based (NSER registered / low income),All provinces,Undergraduate,Announced yearly via HEC (check portal),https://ehsaas.hec.gov.pk,Largest need-based program in Pakistan; apply ...
1,2,HEC Need-Based Undergraduate Scholarship (UGSP),Government - HEC,100% tuition + stipend,All disciplines,Financial need assessed by university committee,All provinces (public HEC-recognized universit...,Undergraduate,Rolling / per semester,https://scholarship.hec.gov.pk,Only public sector universities; private/self-...
2,3,HEC Indigenous PhD Fellowship,Government - HEC,Monthly stipend + research funding,All disciplines,MPhil/MS with research aptitude,All provinces,PhD,Varies by cycle,https://hec.gov.pk/scholarships,Study within Pakistan at HEC-recognized univer...
3,4,HEC Overseas PhD Scholarships (various countries),Government - HEC,Full tuition + stipend abroad,All disciplines,"Strong academic record, MS/MPhil",All provinces,PhD,Varies,https://hec.gov.pk/scholarships,"Includes China, Hungary, France, Austria and o..."
4,5,Benazir Undergraduate Scholarship,Government (BISP-linked),Tuition support + stipend,All disciplines,Underprivileged/low-income background,All provinces,Undergraduate,Announced yearly,https://hec.gov.pk/scholarships,Jointly run with BISP for extremely low-income...


In [53]:
text_columns = [
    "name",
    "provider",
    "coverage",
    "field_requirement_raw",
    "eligibility_raw",
    "domicile_requirement_raw",
    "degree_level_raw",
    "deadline_raw",
    "official_apply_link",
    "notes"
]

for column in text_columns:
    opportunities[column] = (
        opportunities[column]
        .astype("string")
        .str.strip()
    )

In [54]:
def normalize_domicile(value):

    value = str(value).lower().strip()

    if "all province" in value:
        return "any"

    if "punjab" in value:
        return "punjab"

    if "balochistan" in value:
        return "balochistan"

    if "sindh" in value:
        return "sindh"

    if "khyber pakhtunkhwa" in value or "kpk" in value:
        return "kpk"

    if "gb" in value or "gilgit" in value:
        return "gb"

    if "ajk" in value:
        return "ajk"

    if "fata" in value:
        return "fata"

    return "unknown"

In [55]:
opportunities["domicile_requirement"] = (
    opportunities["domicile_requirement_raw"]
    .apply(normalize_domicile)
)

In [57]:
def normalize_degree(value):

    value = str(value).lower().strip()

    if "any level" in value:
        return "any"

    if value == "varies":
        return "unknown"

    if "undergraduate" in value:
        return "bachelors"

    if "master" in value:
        return "masters"

    if "phd" in value or "dphil" in value:
        return "phd"

    if "intermediate" in value:
        return "intermediate"

    if "school" in value:
        return "school"

    return "unknown"

In [58]:
opportunities["degree_level"] = (
    opportunities["degree_level_raw"]
    .apply(normalize_degree)
)

In [59]:
def extract_degree_levels(value):

    value = str(value).lower()

    levels = []

    if "school" in value:
        levels.append("school")

    if "intermediate" in value:
        levels.append("intermediate")

    if "undergraduate" in value:
        levels.append("bachelors")

    if "master" in value:
        levels.append("masters")

    if "phd" in value or "dphil" in value:
        levels.append("phd")

    if "any level" in value:
        levels = ["any"]

    if not levels:
        levels = ["unknown"]

    return levels

In [60]:
opportunities["supported_degree_levels"] = (
    opportunities["degree_level_raw"]
    .apply(extract_degree_levels)
)

In [61]:
def normalize_field(value):

    value = str(value).lower().strip()

    if "all discipline" in value or "all field" in value:
        return ["any"]

    fields = []

    if "computer science" in value or "cs" in value:
        fields.append("computer science")

    if "it" in value:
        fields.append("information technology")

    if "data" in value:
        fields.append("data science")

    if "engineering" in value:
        fields.append("engineering")

    if "business" in value:
        fields.append("business")

    if "economics" in value:
        fields.append("economics")

    if "social science" in value:
        fields.append("social sciences")

    if "health" in value:
        fields.append("health")

    if "medical" in value:
        fields.append("medicine")

    if "agriculture" in value:
        fields.append("agriculture")

    if "governance" in value:
        fields.append("governance")

    if "climate" in value:
        fields.append("climate")

    if not fields:
        fields.append("unknown")

    return list(set(fields))

In [17]:
results_df["match_score"] = (
    results_df["hard_matches"] * 20
    + results_df["partial_matches"] * 10
)

In [18]:
results_df["match_percentage"] = (
    results_df["match_score"] / 90 * 100
).round(1)

In [19]:
recommended_results = results_df[
    results_df["status"] != "Not Eligible"
].copy()

In [20]:
recommended_results = recommended_results.sort_values(
    by="match_score",
    ascending=False
)

In [21]:
recommended_results[
    [
        "opportunity_id",
        "opportunity_name",
        "hard_matches",
        "partial_matches",
        "match_percentage",
        "status"
    ]
].head(10)

,opportunity_id,opportunity_name,hard_matches,partial_matches,match_percentage,status
0,1,DAAD Scholarship,3,3,100.0,Eligible
3,4,TU Munich CS Program,3,3,100.0,Eligible
2,3,Punjab Merit Scholarship,2,2,66.7,Partial Match


In [23]:
recommended_results = recommended_results.merge(
    df[
        [
            "opportunity_id",
            "country",
            "funding_type",
            "deadline"
        ]
    ],
    on="opportunity_id",
    how="left"
)

In [24]:
final_recommendations = recommended_results[
    [
        "opportunity_id",
        "opportunity_name",
        "country",
        "funding_type",
        "deadline",
        "hard_matches",
        "partial_matches",
        "match_percentage",
        "status"
    ]
]

In [25]:
final_recommendations.head(10)

,opportunity_id,opportunity_name,country,funding_type,deadline,hard_matches,partial_matches,match_percentage,status
0,1,DAAD Scholarship,Germany,Full,2026-11-30,3,3,100.0,Eligible
1,4,TU Munich CS Program,Germany,Self-fund,2027-01-15,3,3,100.0,Eligible
2,3,Punjab Merit Scholarship,Pakistan,Partial,2026-09-20,2,2,66.7,Partial Match
